## Method

1. **Build a participant pool** — the raw ARFF has one row per *date*, so the same
   person appears many times. We deduplicate on the personality + interest feature
   vector to recover one row per unique person.
2. **Cluster each gender's pool once** with K-means (k=6) on standardized personality
   features **and** hobby features, weighted so each group contributes equally to
   the distance metric (see "Equal-group weighting" below).
3. **Stratified split** — for each cluster, randomly split the members into Set A
   and Set B halves. This guarantees A and B have the same cluster distribution
   (good comparability) **and** are non-overlapping by construction.
4. **Sample 30** — from each of A_pool and B_pool, take 5 per cluster to reach 30.

Mede mogelijk gemaakt door Claude :)


In [1]:
# load modules and config
import os
import pandas as pd
import numpy as np
from scipy.io import arff
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split


ARFF_PATH      = 'speeddating.arff'   
OUTPUT_DIR     = 'profile_sets'       # CSVs land here (relative path: works on every machine)
N_PER_SET      = 30
N_CLUSTERS     = 6                    # 30 / 5 per cluster
PER_CLUSTER    = N_PER_SET // N_CLUSTERS
RANDOM_STATE   = 42                   # one seed, everything reproducible

# Age filter: participants are 20-30. The dataset goes up to 55, but only ~9 profiles >35.
# Set to None to keep all profiles; set to (low, high) to restrict.
AGE_FILTER     = None                 # e.g. (20, 35) if you want closer to participant age range

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load data

In [2]:
# view data
arff_file = arff.loadarff(ARFF_PATH)
df = pd.DataFrame(arff_file[0])

# decode bytes to string for gender to split the datasets
df['gender'] = df['gender'].str.decode('utf-8')

RATING_COLS = ['sincere', 'intelligence', 'funny', 'ambition',
               'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art',
               'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater',
               'movies', 'concerts', 'music', 'shopping', 'yoga']
df[RATING_COLS] = df[RATING_COLS].clip(lower=0, upper=10) # with upper and lower limit because there were some over 10


## Get the participant pool

In [3]:
PERSONALITY_FEATURES = ['sincere', 'intelligence', 'funny', 'ambition']

HOBBY_FEATURES = [
    'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art',
    'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater',
    'movies', 'concerts', 'music', 'shopping', 'yoga',
]

# columns used to identify a unique participant
# 'wave' is used here to keep them as seperate people even if they have the same ratings
ID_COLS = ['wave', 'gender', 'age'] + PERSONALITY_FEATURES + HOBBY_FEATURES 

participants = (
    df[ID_COLS]
      .drop_duplicates()
      .dropna()
      .reset_index(drop=True)
)
participants['ID'] = participants.index  # make our own ID

# OPTIONAL AGE FILTER---------------------------------------------------------------------
if AGE_FILTER is not None:
    lo, hi = AGE_FILTER
    before = len(participants)
    participants = participants[participants['age'].between(lo, hi)].reset_index(drop=True)
    print(f"Age filter {AGE_FILTER}: kept {len(participants)} / {before} participants")
#------------------------------------------------------------------------------------------

# create two df's of males and females
females = participants[participants['gender'] == 'female'].reset_index(drop=True)
males   = participants[participants['gender'] == 'male'].reset_index(drop=True)

print(f"Unique participants — Female: {len(females)}, Male: {len(males)}")
print(f"Age range: {participants['age'].min():.0f}–{participants['age'].max():.0f}  "
      f"(median {participants['age'].median():.0f})")

Unique participants — Female: 267, Male: 273
Age range: 18–55  (median 26)


## Cluster on each gender dataframe with equal weighting

In [4]:
def build_weighted_features(pool: pd.DataFrame) -> np.ndarray:
    """
    Z-score personality and hobby features separately, then scale each group by
    1/sqrt(group_size) so the two groups contribute equal variance to distance.
    """
    p = StandardScaler().fit_transform(pool[PERSONALITY_FEATURES])
    h = StandardScaler().fit_transform(pool[HOBBY_FEATURES])

    p_weighted = p / np.sqrt(len(PERSONALITY_FEATURES))   # / sqrt(4)
    h_weighted = h / np.sqrt(len(HOBBY_FEATURES))         # / sqrt(17)

    return np.hstack([p_weighted, h_weighted])

def cluster_and_split(pool: pd.DataFrame, random_state: int):
    """
    Cluster `pool` into N_CLUSTERS on equal-weighted personality + hobby features,
    then split each cluster stratified 50/50 into A and B pools using sklearn's
    train_test_split (stratifying on cluster keeps the cluster proportions equal
    in both halves).
    Returns: (pool_with_cluster, A_pool, B_pool)
    """
    pool = pool.copy()

    X = build_weighted_features(pool)
    km = KMeans(n_clusters=N_CLUSTERS, random_state=random_state, n_init=10)
    pool['cluster'] = km.fit_predict(X)

    a_pool, b_pool = train_test_split(
        pool,
        test_size=0.5,
        stratify=pool['cluster'],
        random_state=random_state,
    )

    return pool, a_pool, b_pool

def pick_30(pool_half: pd.DataFrame, random_state: int) -> pd.DataFrame:
    """Take PER_CLUSTER (=5) profiles from each of N_CLUSTERS clusters → 30 total."""
    picked = []
    for c in range(N_CLUSTERS):
        group = pool_half[pool_half['cluster'] == c]
        picked.append(group.sample(n=PER_CLUSTER, random_state=random_state))
    return pd.concat(picked).reset_index(drop=True)

In [5]:
# female profile sets
females_clustered, female_A_pool, female_B_pool = cluster_and_split(females, RANDOM_STATE)
female_A = pick_30(female_A_pool, RANDOM_STATE)
female_B = pick_30(female_B_pool, RANDOM_STATE + 1)

# male profile sets
males_clustered, male_A_pool, male_B_pool = cluster_and_split(males, RANDOM_STATE)
male_A = pick_30(male_A_pool, RANDOM_STATE)
male_B = pick_30(male_B_pool, RANDOM_STATE + 1)

print("Set sizes:")
print(f"  Female A: {len(female_A)}   Female B: {len(female_B)}")
print(f"  Male A:   {len(male_A)}     Male B:   {len(male_B)}")

Set sizes:
  Female A: 30   Female B: 30
  Male A:   30     Male B:   30


## Save into CSV files

In [ ]:
OUTPUT_COLS = ['age'] + PERSONALITY_FEATURES + HOBBY_FEATURES

female_A[OUTPUT_COLS].to_csv(f'{OUTPUT_DIR}/female_set_A.csv', index=True)
female_B[OUTPUT_COLS].to_csv(f'{OUTPUT_DIR}/female_set_B.csv', index=True)
male_A[OUTPUT_COLS].to_csv(f'{OUTPUT_DIR}/male_set_A.csv',     index=True)
male_B[OUTPUT_COLS].to_csv(f'{OUTPUT_DIR}/male_set_B.csv',     index=True)

print(f"Saved 4 files to {OUTPUT_DIR}/")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fn}")

Saved 4 files to profile_sets/
  female_set_A.csv
  female_set_B.csv
  male_set_A.csv
  male_set_B.csv
